# Lekcija 27: Kalibracija kamere

Lekcija 26 uvela je matricu unutarnjih parametara kamere $K$ za modeliranje kamere s malim otvorom. Klasičan pristup **kalibraciji kamere** (<a href="../references.html#zhang-2000">Zhang, 2000.</a>) određuje $K$ tako da kamera promatra poznati ravni uzorak (šahovnicu) iz nekoliko različitih kutova: izvlači se homografija (lekcija 24) između ravnine šahovnice i svake slike, a zatim se ograničenja iz više takvih homografija kombiniraju kako bi se procijenio $K$.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Što kalibracija određuje

Kalibracija kamere određuje dvije zasebne stvari:

1. **Unutarnje parametre** $K = \begin{bmatrix}f_x & s & c_x\\0 & f_y & c_y\\0&0&1\end{bmatrix}$: žarišne duljine, glavnu točku i skošenje — idealni dio modela kamere s malim otvorom koji se koristi u lekcijama 28 i 29. Za stvarne kamere sigurno se može pretpostaviti da je skošenje nula.
2. **Koeficijente distorzije** $(k_1, k_2, p_1, p_2, k_3, \dots)$: stvarne leće nisu savršeni mali otvori. Radijalna distorzija ($k_1, k_2, k_3$) savija ravne linije u krivulje (bačvasta ili jastučasta distorzija, ovisno o predznaku); tangencijalna distorzija ($p_1, p_2$) opisuje činjenicu da leća nije savršeno paralelna sa senzorom. Svaka jednadžba u lekcijama 28 i 29 prešutno pretpostavlja model kamere s malim otvorom (odnosno da je distorzija već uklonjena, što zahtijeva kalibraciju).

## Sintetički kalibracijski postav

Stvaramo sintetičke slike kamere koja promatra šahovnicu: definiramo stvarni $K$ i distorziju, definiramo 3D položaje kutova ravne šahovnice i generiramo nekoliko „fotografija” iz različitih položaja i orijentacija projekcijom kutova pomoću `cv2.projectPoints` (koji primjenjuje distorziju upravo kao stvarna leća). Time dobivamo istu vrstu ulaza koju `cv2.calibrateCamera` očekuje od stvarno detektiranih kutova šahovnice &mdash; uz poznate stvarne vrijednosti s kojima možemo usporediti rezultat.

In [ ]:
rng = np.random.default_rng(0)

cols, rows = 9, 6  # internal corners of a 10x7-square checkerboard
corners_3d = np.zeros((rows * cols, 3), dtype=np.float64)
corners_3d[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
corners_3d[:, :2] -= corners_3d[:, :2].mean(axis=0)  # center the board on its own origin

K_true = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1]], dtype=np.float64)
dist_true = np.array([-0.3, 0.1, 0.001, -0.0005, 0.02])  # k1, k2, p1, p2, k3
image_size = (640, 480)

object_points, image_points = [], []
for _ in range(40):
    rvec = rng.uniform(-0.4, 0.4, 3)                                     # a somewhat random tilt
    tvec = np.array([rng.uniform(-0.5, 0.5), rng.uniform(-0.4, 0.4), rng.uniform(9, 13)])
    projected, _ = cv2.projectPoints(corners_3d, rvec, tvec, K_true, dist_true)
    projected = projected.reshape(-1, 2)
    if np.all(projected >= 5) and np.all(projected[:, 0] < image_size[0] - 5) and np.all(projected[:, 1] < image_size[1] - 5):
        object_points.append(corners_3d.astype(np.float32))
        image_points.append(projected.astype(np.float32))

print(f'{len(object_points)} usable synthetic views (out of 40 attempted; some fell outside the frame)')

### Iscrtavanje sintetičkih fotografija

Kako bismo imali na čemu detektirati kutove, a ne samo koordinate točaka, trebaju nam stvarne slike. Budući da je šahovnica ravna, preslikavanje njezina vlastitog 2D površinskog uzorka u fotografiju upravo je homografija. Stoga deformiranje slike šahovnice (izmjeničnih crnih i bijelih kvadrata) homografijom svakog pogleda sintetizira fotografiju &mdash; kao zamjenu za stvarnu fotografiju.

In [ ]:
sq = 60  # pixels per square in the flat target image
board = np.zeros(((rows + 1) * sq, (cols + 1) * sq), dtype=np.uint8)
for r in range(rows + 1):
    for c in range(cols + 1):
        if (r + c) % 2 == 0:
            board[r * sq:(r + 1) * sq, c * sq:(c + 1) * sq] = 255
board_bgr = cv2.cvtColor(board, cv2.COLOR_GRAY2BGR)

# the internal corners' locations in the flat target's own pixel coordinates,
# in the same order as corners_3d
board_pts = np.array([[(c + 1) * sq, (r + 1) * sq] for r in range(rows) for c in range(cols)],
                      dtype=np.float32)

photos = []
for img_pts in image_points:
    H, _ = cv2.findHomography(board_pts, img_pts)
    photos.append(cv2.warpPerspective(board_bgr, H, image_size, borderValue=(128, 128, 128)))

### Detekcija kalibracijskog uzorka

Stvarni kalibracijski postupak zna ravni 2D raspored šahovnice (fizičku veličinu kvadrata, a time i ravninsku mrežu `corners_3d`) &mdash; ali ne zna položaj i orijentaciju šahovnice u 3D prostoru u odnosu na kameru pri svakoj snimci (rotaciju i translaciju), niti 2D pikselne položaje njezinih kutova na fotografijama. Zato je prvi korak svakog kalibracijskog postupka pronalaženje tih 2D položaja: pokrećemo OpenCV-jev `cv2.findChessboardCorners` na svakoj fotografiji &mdash; upravo kao na stvarnoj fotografiji &mdash; i uspoređujemo detekcije s poznatim stvarnim vrijednostima `image_points`.

In [ ]:
detected_object_points, detected_image_points = [], []
corner_errors = []
example_photo = example_corners_raw = None

for obj_pts, img_pts, photo in zip(object_points, image_points, photos):
    gray = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)

    found, corners = cv2.findChessboardCorners(gray, (cols, rows))
    if not found:
        continue
    corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1),
                                (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))

    if example_photo is None:
        example_photo, example_corners_raw = photo, corners

    corners = corners.reshape(-1, 2)[::-1]  # cv2 scans from the opposite corner vs. our board_pts order
    detected_object_points.append(obj_pts)
    detected_image_points.append(corners.astype(np.float32))
    corner_errors.append(np.linalg.norm(corners - img_pts, axis=1).mean())

print(f'corners detected in {len(detected_image_points)} / {len(image_points)} synthetic photos')
print(f'mean per-view corner error vs. ground truth: {np.mean(corner_errors):.2f} px '
      f'(max {np.max(corner_errors):.2f} px)')

example_annotated = example_photo.copy()
cv2.drawChessboardCorners(example_annotated, (cols, rows), example_corners_raw, True)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(board, cmap='gray')
axes[0].set_title('Flat checkerboard target')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(example_annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('cv2.findChessboardCorners,\none synthetic photo')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Provedba kalibracije

Zhangov kalibracijski algoritam (`cv2.calibrateCamera`) uzima poznate položaje kutova na šahovnici i njihove *detektirane* 2D položaje na slikama iz svih pogleda te zajednički određuje $K$, distorziju i položaj i orijentaciju svakog pogleda.

In [ ]:
rms_error, K_estimated, dist_estimated, rvecs, tvecs = cv2.calibrateCamera(
    detected_object_points, detected_image_points, image_size, None, None)

print(f'RMS reprojection error: {rms_error:.2e} pixels\n')
print('K (true):\n', K_true)
print('K (estimated):\n', np.round(K_estimated, 3))
print()
print('distortion (true):     ', dist_true)
print('distortion (estimated):', np.round(dist_estimated.ravel(), 5))

Uz detekcije kutova koje sadrže šum, kalibracija ne određuje stvarne parametre potpuno točno. Kvalitetu kalibracije mjerimo pogreškom reprojekcije. Zato u praksi treba koristiti što više pogleda iz što raznovrsnijih kutova: više (i raznovrsnija) ograničenja usrednjavaju šum detektora i bolje ograničavaju procjenu.

## Ravne linije koje leća savija

Distorzija leće pretvara ravne linije u krivulje. U normaliziranim koordinatama kamere (prije primjene $K$):

$$x_d = x(1 + k_1r^2 + k_2r^4 + k_3r^6) + 2p_1xy + p_2(r^2+2x^2)$$
$$y_d = y(1 + k_1r^2 + k_2r^4 + k_3r^6) + p_1(r^2+2y^2) + 2p_2xy$$

gdje je $r^2 = x^2+y^2$. To je upravo model koji je `cv2.projectPoints` gore interno primijenio.

In [ ]:
def distort_normalized(xy, dist):
    k1, k2, p1, p2, k3 = dist
    x, y = xy[:, 0], xy[:, 1]
    r2 = x**2 + y**2
    radial = 1 + k1 * r2 + k2 * r2**2 + k3 * r2**3
    xd = x * radial + 2 * p1 * x * y + p2 * (r2 + 2 * x**2)
    yd = y * radial + p1 * (r2 + 2 * y**2) + 2 * p2 * x * y
    return np.stack([xd, yd], axis=1)

def make_grid_lines():
    lines = []
    for xv in np.linspace(-0.38, 0.38, 9):
        ys = np.linspace(-0.28, 0.28, 50)
        lines.append(np.stack([np.full_like(ys, xv), ys], axis=1))
    for yv in np.linspace(-0.28, 0.28, 7):
        xs = np.linspace(-0.38, 0.38, 50)
        lines.append(np.stack([xs, np.full_like(xs, yv)], axis=1))
    return lines

def draw_lines(lines_px, height=480, width=640):
    img = np.full((height, width, 3), 255, dtype=np.uint8)
    for line in lines_px:
        for i in range(len(line) - 1):
            cv2.line(img, tuple(line[i].astype(int)), tuple(line[i + 1].astype(int)), (0, 0, 0), 1)
    return img

grid_lines_normalized = make_grid_lines()

straight_px = [line @ K_true[:2, :2].T + K_true[:2, 2] for line in grid_lines_normalized]
distorted_px = [distort_normalized(line, dist_true) @ K_true[:2, :2].T + K_true[:2, 2]
                for line in grid_lines_normalized]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(draw_lines(straight_px))
axes[0].set_title('Ideal pinhole (no distortion)')
axes[1].imshow(draw_lines(distorted_px))
axes[1].set_title(f'Through the lens\n(k1={dist_true[0]}, k2={dist_true[1]})')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Uklanjanje distorzije procijenjenim parametrima

`cv2.undistortPoints` invertira model distorzije, pretvarajući distorzirane pikselne koordinate natrag u normalizirane koordinate bez distorzije. Primjena na gore savijene linije mreže, uz našu *procijenjenu* (a ne stvarnu) kalibraciju, trebala bi ih ponovno izravnati.

In [ ]:
undistorted_normalized = [
    cv2.undistortPoints(px.reshape(-1, 1, 2).astype(np.float64), K_estimated, dist_estimated).reshape(-1, 2)
    for px in distorted_px
]
undistorted_px = [line @ K_estimated[:2, :2].T + K_estimated[:2, 2] for line in undistorted_normalized]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(draw_lines(distorted_px))
axes[0].set_title('Distorted (as captured)')
axes[1].imshow(draw_lines(undistorted_px))
axes[1].set_title('Undistorted using estimated K, dist')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

original_normalized = np.concatenate(grid_lines_normalized)
recovered_normalized = np.concatenate(undistorted_normalized)
print(f'max recovery error (normalized coords): {np.abs(original_normalized - recovered_normalized).max():.2e}')

Linije se izravnavaju, iako ne savršeno &mdash; ostaje blago zakrivljenje jer su i `K_estimated`/`dist_estimated` samo približno točni. Ovaj postupak kalibracije stvarne kamere trebalo bi, ako je moguće, provesti prije predaje stvarnih fotografija geometrijskim postupcima iz lekcija 28 i 29, koji svi prešutno pretpostavljaju idealan model kamere s malim otvorom, bez distorzije. Ipak, imajte na umu da mnogi suvremeni postupci provode automatsku kalibraciju bez slika šahovnice, uz pojednostavnjene modele kamere (obično samo jedan parametar žarišne duljine &mdash; zanemarujući omjer stranica, glavnu točku izvan središta i distorziju leće).

### Zadaci

1. Prije kalibracije dodajte dodatni sintetički pikselni šum (npr. `rng.normal(0, 0.5, corners.shape)`) detektiranim kutovima svakog pogleda, povrh stvarnog šuma detektora. Koliko raste RMS pogreška reprojekcije i koliko se `K_estimated` dodatno udaljava od `K_true`?
2. Ovaj postav završava sa samo nekoliko uspješno detektiranih pogleda (ispisanih gore). Pokušajte prije kalibracije smanjiti `detected_object_points`/`detected_image_points` na samo 3 pogleda. Pogoršava li se procjena primjetno, osobito za koeficijente distorzije (koji za dobro ograničenu procjenu zahtijevaju kutove raspoređene preko cijelog kadra, uključujući blizu rubova)?
3. Postavite `dist_true = np.zeros(5)` (savršeni mali otvor, bez distorzije) i ponovno pokrenite cijeli notebook. Potvrdite da „distorzirana” i „idealna” mreža postaju identične te da kalibracija i dalje razumno dobro određuje `K_true`, čak i kada treba procijeniti nultu distorziju.